# IBKR Options — Parquet Explorer

Quick notebook to load and inspect every Parquet dataset written by the pipeline into `data/`.

In [13]:
import os
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

In [16]:
# Root of the repo — works regardless of where the notebook is run from
REPO_ROOT = Path(__file__).parent.parent if "__file__" in dir() else Path().resolve().parent
DATA_DIR = REPO_ROOT / "data"

DATASETS = [
    "option_chain",
    "spreads",
    "iron_condors",
    "iron_butterflies",
    "calendars",
    "strangles",
    "expected_move",
    "max_pain",
]

def load(name: str) -> pd.DataFrame | None:
    path = DATA_DIR / name
    if not path.exists():
        print(f"⚠️  {name}: not found at {path}")
        return None
    df = pd.read_parquet(path)
    print(f"✅  {name}: {len(df):,} rows × {len(df.columns)} cols")
    return df

print(f"DATA_DIR = {DATA_DIR.resolve()}\n")
dfs = {name: load(name) for name in DATASETS}

DATA_DIR = /Users/gabriel/Desktop/IBKR/data

✅  option_chain: 100 rows × 18 cols
✅  spreads: 439 rows × 27 cols
✅  iron_condors: 5 rows × 26 cols
✅  iron_butterflies: 22 rows × 29 cols
✅  calendars: 128 rows × 23 cols
✅  strangles: 154 rows × 25 cols
✅  expected_move: 5 rows × 14 cols
✅  max_pain: 4 rows × 9 cols


## Option Chain

In [12]:
# ── Bid/Ask/Close quality diagnostic ────────────────────────────────────────
# IBKR returns -1 for bid/ask when the market is closed (documented behaviour:
# https://interactivebrokers.github.io/tws-api/md_receive.html).
# The pipeline converts -1 → None via _safe_float, so 'mid' is derived from
# 'close' as a fallback.  Bid/ask will populate correctly during market hours.
df = dfs["option_chain"]
if df is not None:
    total = len(df)
    def pct(n): return f"{n:,}  ({n/total*100:.1f}%)"

    null_bid   = df["bid"].isna().sum()
    null_ask   = df["ask"].isna().sum()
    null_mid   = df["mid"].isna().sum()
    null_delta = df["delta"].isna().sum()

    has_close = "close" in df.columns
    null_close = df["close"].isna().sum() if has_close else total

    print(f"Total contracts : {total:,}")
    print(f"  bid    null   : {pct(null_bid)}")
    print(f"  ask    null   : {pct(null_ask)}")
    if has_close:
        print(f"  close  null   : {pct(null_close)}")
    else:
        print("  close         : ⚠️  column missing — re-run pipeline to pick up schema change")
    print(f"  mid    null   : {pct(null_mid)}")
    print(f"  delta  null   : {pct(null_delta)}")
    print()

    if null_bid / total > 0.5:
        print("⚠️  Most bid/ask values are null — data was likely collected outside")
        print("   US market hours (9:30–16:00 ET). IBKR returns -1 (→ None) for")
        print("   bid/ask when no live/delayed quote is available.")
        if has_close and null_close / total < 0.5:
            print(f"   ✅ 'close' is populated for {total - null_close:,}/{total:,} contracts")
            print("      and is used as the 'mid' fallback outside market hours.")
    else:
        print("✅  Bid/ask data looks healthy.")

Total contracts : 100
  bid    null   : 0  (0.0%)
  ask    null   : 0  (0.0%)
  close         : ⚠️  column missing — re-run pipeline to pick up schema change
  mid    null   : 0  (0.0%)
  delta  null   : 100  (100.0%)

✅  Bid/ask data looks healthy.


In [17]:
df = dfs["option_chain"]
if df is not None:
    display(df.head(10))
    print("\n--- dtypes ---")
    display(df.dtypes.to_frame("dtype"))
    print("\n--- describe ---")
    display(df.describe())

,symbol,expiration,strike,right,bid,ask,last,close,mid,delta,gamma,theta,vega,implied_vol,open_interest,volume,underlying_price,dte
0,DKNG,2026-04-24,19.0000,C,-1.0000,-1.0000,NaN,3.5900,3.5900,0.8968,0.0457,-0.0195,0.0110,0.6932,0.0000,0.0000,23.0800,21
1,DKNG,2026-04-24,19.0000,P,-1.0000,-1.0000,NaN,0.3600,0.3600,-0.1007,0.0449,-0.0170,0.0110,0.6932,76.0000,0.0000,23.0800,21
2,DKNG,2026-04-24,20.0000,C,-1.0000,-1.0000,NaN,2.7300,2.7300,0.8471,0.0649,-0.0223,0.0138,0.6418,76.0000,0.0000,23.0800,21
3,DKNG,2026-04-24,20.0000,P,-1.0000,-1.0000,NaN,0.5000,0.5000,-0.1496,0.0639,-0.0200,0.0138,0.6418,292.0000,0.0000,23.0800,21
4,DKNG,2026-04-24,21.0000,C,-1.0000,-1.0000,NaN,2.0000,2.0000,0.7759,0.0888,-0.0246,0.0167,0.5948,87.0000,0.0000,23.0800,21
5,DKNG,2026-04-24,21.0000,P,-1.0000,-1.0000,NaN,0.7600,0.7600,-0.2199,0.0877,-0.0222,0.0167,0.5948,64.0000,0.0000,23.0800,21
6,DKNG,2026-04-24,22.0000,C,-1.0000,-1.0000,NaN,1.3800,1.3800,0.6815,0.1137,-0.0284,0.0210,0.5547,143.0000,0.0000,23.0800,21
7,DKNG,2026-04-24,22.0000,P,-1.0000,-1.0000,NaN,1.1400,1.1400,-0.3166,0.1138,-0.0258,0.0210,0.5547,89.0000,0.0000,23.0800,21
8,DKNG,2026-04-24,23.0000,C,-1.0000,-1.0000,NaN,0.9100,0.9100,0.5577,0.1330,-0.0284,0.0224,0.5233,509.0000,0.0000,23.0800,21
9,DKNG,2026-04-24,23.0000,P,-1.0000,-1.0000,NaN,1.6700,1.6700,-0.4408,0.1335,-0.0259,0.0225,0.5233,50.0000,0.0000,23.0800,21



--- dtypes ---


,dtype
symbol,str
expiration,object
strike,float64
right,str
bid,float64
ask,float64
last,float64
close,float64
mid,float64
delta,float64



--- describe ---


,strike,bid,ask,last,close,mid,delta,gamma,theta,vega,implied_vol,open_interest,volume,underlying_price,dte
count,100.0000,100.0000,100.0000,0.0000,100.0000,100.0000,10.0000,10.0000,10.0000,10.0000,10.0000,100.0000,100.0000,100.0000,100.0000
mean,23.0600,-1.0000,-1.0000,NaN,2.1322,2.1322,0.2531,0.0890,-0.0234,0.0170,0.6016,235.2300,0.0000,23.0800,37.1000
std,2.6811,0.0000,0.0000,NaN,1.4654,1.4654,0.5410,0.0336,0.0039,0.0045,0.0639,799.5773,0.0000,0.0000,10.9217
min,18.5000,-1.0000,-1.0000,NaN,0.1200,0.1200,-0.4408,0.0449,-0.0284,0.0110,0.5233,0.0000,0.0000,23.0800,21.0000
25%,21.0000,-1.0000,-1.0000,NaN,0.9075,0.9075,-0.2023,0.0641,-0.0258,0.0138,0.5547,0.0000,0.0000,23.0800,28.0000
50%,23.0000,-1.0000,-1.0000,NaN,1.8200,1.8200,0.2285,0.0882,-0.0235,0.0167,0.5948,3.0000,0.0000,23.0800,35.0000
75%,25.0000,-1.0000,-1.0000,NaN,3.2225,3.2225,0.7523,0.1138,-0.0205,0.0210,0.6418,76.0000,0.0000,23.0800,49.0000
max,27.5000,-1.0000,-1.0000,NaN,5.7200,5.7200,0.8968,0.1335,-0.0170,0.0225,0.6932,4932.0000,0.0000,23.0800,49.0000


## Credit Spreads &amp; Debit Spreads

In [18]:
df = dfs["spreads"]
if df is not None:
    print("Strategy counts:")
    display(df["strategy"].value_counts())
    print("\n--- Credit Spreads (top 10 by spread_ratio) ---")
    display(
        df[df["strategy"].str.contains("Credit")]
        .sort_values("spread_ratio", ascending=False)
        .head(10)
    )
    print("\n--- Debit Spreads (top 10 by spread_ratio) ---")
    display(
        df[df["strategy"].str.contains("Debit")]
        .sort_values("spread_ratio", ascending=False)
        .head(10)
    )

Strategy counts:


strategy
Put Debit Spread      154
Call Debit Spread     154
Put Credit Spread      76
Call Credit Spread     55
Name: count, dtype: int64


--- Credit Spreads (top 10 by spread_ratio) ---


,symbol,expiration,right,strategy,underlying_price,spread_ratio,short_strike,pct_to_short_strike,short_mid,short_delta,long_strike,pct_to_long_strike,long_mid,long_delta,credit,width,max_profit,max_loss,credit_yield,roc,breakeven,pct_to_breakeven,dte,net_delta,net_gamma,net_theta,net_vega
105,DKNG,2026-05-08,P,Put Credit Spread,23.0800,55.0000,23.0000,0.3466,2.2000,NaN,22.0000,4.6794,1.6500,NaN,55.0000,1.0000,55.0000,45.0000,122.2222,2.3913,22.4500,2.7296,35,NaN,NaN,NaN,NaN
126,DKNG,2026-04-24,P,Put Credit Spread,23.0800,53.0000,23.0000,0.3466,1.6700,-0.4408,22.0000,4.6794,1.1400,-0.3166,53.0000,1.0000,53.0000,47.0000,112.7660,2.3043,22.4700,2.6430,21,-0.1241,0.0197,-0.0001,0.0015
137,DKNG,2026-05-01,P,Put Credit Spread,23.0800,52.0000,23.0000,0.3466,1.8700,NaN,22.0000,4.6794,1.3500,NaN,52.0000,1.0000,52.0000,48.0000,108.3333,2.2609,22.4800,2.5997,28,NaN,NaN,NaN,NaN
139,DKNG,2026-05-22,P,Put Credit Spread,23.0800,52.0000,23.0000,0.3466,2.4800,NaN,22.5000,2.5130,2.2200,NaN,26.0000,0.5000,26.0000,24.0000,108.3333,1.1304,22.7400,1.4731,49,NaN,NaN,NaN,NaN
151,DKNG,2026-05-22,P,Put Credit Spread,23.0800,51.0000,23.0000,0.3466,2.4800,NaN,22.0000,4.6794,1.9700,NaN,51.0000,1.0000,51.0000,49.0000,104.0816,2.2174,22.4900,2.5563,49,NaN,NaN,NaN,NaN
158,DKNG,2026-05-22,P,Put Credit Spread,23.0800,50.0000,22.5000,2.5130,2.2200,NaN,22.0000,4.6794,1.9700,NaN,25.0000,0.5000,25.0000,25.0000,100.0000,1.1111,22.2500,3.5962,49,NaN,NaN,NaN,NaN
169,DKNG,2026-05-22,P,Put Credit Spread,23.0800,48.6667,23.0000,0.3466,2.4800,NaN,21.5000,6.8458,1.7500,NaN,73.0000,1.5000,73.0000,77.0000,94.8052,3.1739,22.2700,3.5095,49,NaN,NaN,NaN,NaN
181,DKNG,2026-05-08,P,Put Credit Spread,23.0800,48.0000,23.0000,0.3466,2.2000,NaN,21.0000,9.0121,1.2400,NaN,96.0000,2.0000,96.0000,104.0000,92.3077,4.1739,22.0400,4.5061,35,NaN,NaN,NaN,NaN
190,DKNG,2026-05-22,P,Put Credit Spread,23.0800,47.0000,22.5000,2.5130,2.2200,NaN,21.5000,6.8458,1.7500,NaN,47.0000,1.0000,47.0000,53.0000,88.6792,2.0889,22.0300,4.5494,49,NaN,NaN,NaN,NaN
192,DKNG,2026-05-22,P,Put Credit Spread,23.0800,47.0000,23.0000,0.3466,2.4800,NaN,21.0000,9.0121,1.5400,NaN,94.0000,2.0000,94.0000,106.0000,88.6792,4.0870,22.0600,4.4194,49,NaN,NaN,NaN,NaN



--- Debit Spreads (top 10 by spread_ratio) ---


,symbol,expiration,right,strategy,underlying_price,spread_ratio,short_strike,pct_to_short_strike,short_mid,short_delta,long_strike,pct_to_long_strike,long_mid,long_delta,credit,width,max_profit,max_loss,credit_yield,roc,breakeven,pct_to_breakeven,dte,net_delta,net_gamma,net_theta,net_vega
0,DKNG,2026-04-24,P,Put Debit Spread,23.0800,80.7500,23.0000,0.3466,1.6700,-0.4408,27.0000,16.9844,4.9000,NaN,-323.0000,4.0000,77.0000,323.0000,23.8390,14.0435,23.7700,2.9896,21,NaN,NaN,NaN,NaN
1,DKNG,2026-05-01,P,Put Debit Spread,23.0800,77.5000,23.0000,0.3466,1.8700,NaN,27.0000,16.9844,4.9700,NaN,-310.0000,4.0000,90.0000,310.0000,29.0323,13.4783,23.9000,3.5529,28,NaN,NaN,NaN,NaN
2,DKNG,2026-04-24,P,Put Debit Spread,23.0800,76.3333,23.0000,0.3466,1.6700,-0.4408,26.0000,12.6516,3.9600,NaN,-229.0000,3.0000,71.0000,229.0000,31.0044,9.9565,23.7100,2.7296,21,NaN,NaN,NaN,NaN
3,DKNG,2026-04-24,P,Put Debit Spread,23.0800,75.2000,22.0000,4.6794,1.1400,-0.3166,27.0000,16.9844,4.9000,NaN,-376.0000,5.0000,124.0000,376.0000,32.9787,17.0909,23.2400,0.6932,21,NaN,NaN,NaN,NaN
4,DKNG,2026-05-08,P,Put Debit Spread,23.0800,73.0000,23.0000,0.3466,2.2000,NaN,27.0000,16.9844,5.1200,NaN,-292.0000,4.0000,108.0000,292.0000,36.9863,12.6957,24.0800,4.3328,35,NaN,NaN,NaN,NaN
5,DKNG,2026-05-01,P,Put Debit Spread,23.0800,73.0000,23.0000,0.3466,1.8700,NaN,26.0000,12.6516,4.0600,NaN,-219.0000,3.0000,81.0000,219.0000,36.9863,9.5217,23.8100,3.1629,28,NaN,NaN,NaN,NaN
6,DKNG,2026-05-01,P,Put Debit Spread,23.0800,72.4000,22.0000,4.6794,1.3500,NaN,27.0000,16.9844,4.9700,NaN,-362.0000,5.0000,138.0000,362.0000,38.1215,16.4545,23.3800,1.2998,28,NaN,NaN,NaN,NaN
7,DKNG,2026-05-22,P,Put Debit Spread,23.0800,72.0000,23.0000,0.3466,2.4800,NaN,27.5000,19.1508,5.7200,NaN,-324.0000,4.5000,126.0000,324.0000,38.8889,14.0870,24.2600,5.1127,49,NaN,NaN,NaN,NaN
8,DKNG,2026-04-24,P,Put Debit Spread,23.0800,71.5000,23.0000,0.3466,1.6700,-0.4408,25.0000,8.3189,3.1000,NaN,-143.0000,2.0000,57.0000,143.0000,39.8601,6.2174,23.5700,2.1231,21,NaN,NaN,NaN,NaN
9,DKNG,2026-05-22,P,Put Debit Spread,23.0800,70.7500,23.0000,0.3466,2.4800,NaN,27.0000,16.9844,5.3100,NaN,-283.0000,4.0000,117.0000,283.0000,41.3428,12.3043,24.1700,4.7227,49,NaN,NaN,NaN,NaN


## Iron Condors

In [ ]:
df = dfs["iron_condors"]
if df is not None:
    display(df.sort_values("spread_ratio", ascending=False).head(10))

## Iron Butterflies

In [ ]:
df = dfs["iron_butterflies"]
if df is not None:
    display(df.sort_values("spread_ratio", ascending=False).head(10))

## Strangles

In [ ]:
df = dfs["strangles"]
if df is not None:
    display(df.sort_values("total_premium", ascending=False).head(10))

## Calendar Spreads

In [ ]:
df = dfs["calendars"]
if df is not None:
    display(df.sort_values("theta_differential", ascending=False).head(10))

## Analytics — Expected Move &amp; Max Pain

In [ ]:
df_em = dfs["expected_move"]
if df_em is not None:
    print("--- Expected Move ---")
    display(df_em)

df_mp = dfs["max_pain"]
if df_mp is not None:
    print("--- Max Pain ---")
    display(df_mp)

## Missing Values &amp; Data Quality

In [ ]:
for name, df in dfs.items():
    if df is None:
        continue
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if nulls.empty:
        print(f"✅  {name}: no nulls")
    else:
        print(f"⚠️  {name}:")
        display(nulls.to_frame("null_count"))